# F2-M07: Conclusiones de Fase 2

**TFM: Predicción de Abandono Universitario**

| | |
|---|---|
| **Autora** | María José Morte |
| **Email** | mjmorteruiz@uoc.edu (UOC) \| morte@uji.es (UJI) |

---

## Qué hace
Resumen ejecutivo de la Fase 2: métricas clave del dataset, hallazgos
principales, problemas críticos, regla de negocio para calcular abandono,
estado del proyecto y próximos pasos hacia Fase 3.

## Requisitos
- `df_alumno.parquet` en `data/02_processed/`
- Módulos: `src.config`, `src.utils`, `src.html`

## Genera
- `docs/html/fase2/m07_conclusiones.html`
- `docs/html/fase2/graficos/m07_flujo_proyecto.html`

## Flujo
```
... → M06 Evolución → **M07 Conclusiones** → Fase 3
```

## Siguiente
Fase 3: Feature Engineering

In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN DEL ENTORNO
# ============================================================================
# - Detecta entorno (Colab / local)
# - Localiza ROOT buscando src/ (robusto, sin hardcodes)
# - Importa módulos del proyecto
# - Crea directorios de salida
# ============================================================================

import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# --- Detectar entorno y localizar ROOT ---
def _encontrar_root(start: Path) -> Path:
    for parent in [start] + list(start.parents):
        if (parent / 'src').is_dir():
            return parent
    raise FileNotFoundError(
        f"No se encontró carpeta 'src/' subiendo desde {start}. "
        f"Verifica que el notebook está dentro de AU_UJI/"
    )

ROOT = _encontrar_root(Path.cwd())

sys.path.insert(0, str(ROOT))

# --- Imports ---
import pandas as pd
import numpy as np
import plotly.graph_objects as go

from src.config import RUTA_PROCESSED, RUTA_HTML, info_entorno
from src.utils import crear_directorios, formato_numero_es
from src.html import (
    generar_kpis_html,
    generar_seccion_html,
    generar_html_navegacion_completa,
    guardar_html
)
from src.html.render import render_pagina_desde_fichero

# --- Rutas de salida ---
RUTA_FASE2 = RUTA_HTML / 'fase2'
RUTA_GRAFICOS = RUTA_FASE2 / 'graficos'
crear_directorios([RUTA_FASE2, RUTA_GRAFICOS])

info_entorno()

✓ Directorios verificados: 2
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\FF\AU_UJI_v2
✓ 📁 RAW:           C:\FF\AU_UJI_v2\data\00_raw
✓ 📁 INTERIM:       C:\FF\AU_UJI_v2\data\01_interim
✓ 📁 PROCESSED:     C:\FF\AU_UJI_v2\data\02_processed
✓ 📁 FEATURES:      C:\FF\AU_UJI_v2\data\03_features
✓ 📁 AUTOML:        C:\FF\AU_UJI_v2\data\automl
✓ 📁 NOTEBOOKS:     C:\FF\AU_UJI_v2\notebooks
✓ 📄 Excel principal: C:\FF\AU_UJI_v2\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓ ===========================================================================


In [2]:
# ============================================================================
# CELDA 2: CARGAR DATOS Y CALCULAR MÉTRICAS
# ============================================================================

print('=' * 60)
print('CARGANDO DATOS')
print('=' * 60)

df = pd.read_parquet(RUTA_PROCESSED / 'df_alumno.parquet')

# --- Métricas básicas ---
n_filas = len(df)
n_cols = len(df.columns)

# Alumnos únicos
if 'per_id_ficticio' in df.columns:
    n_alumnos = df['per_id_ficticio'].nunique()
else:
    n_alumnos = n_filas

# Filas por alumno (estructura longitudinal)
filas_por_alumno = n_filas / n_alumnos if n_alumnos > 0 else 1

# Cursos académicos
curso_col = None
for col in ['curso_aca', 'curso_aca_id', 'curso_academico']:
    if col in df.columns:
        curso_col = col
        break

if curso_col:
    n_cursos = df[curso_col].nunique()
    curso_min = df[curso_col].min()
    curso_max = df[curso_col].max()
else:
    n_cursos = 0
    curso_min = curso_max = 'N/A'

print(f'✅ Dataset: {n_filas:,} filas × {n_cols} columnas')
print(f'👥 Alumnos únicos: {n_alumnos:,}')
print(f'📅 Cursos: {n_cursos} ({curso_min} - {curso_max})')
print(f'📊 ~{filas_por_alumno:.1f} filas por alumno')

CARGANDO DATOS
✅ Dataset: 109,568 filas × 37 columnas
👥 Alumnos únicos: 30,872
📅 Cursos: 11 (2010 - 2020)
📊 ~3.5 filas por alumno


In [3]:
# ============================================================================
# CELDA 3: ANÁLISIS DE CALIDAD
# ============================================================================

print('=' * 60)
print('ANÁLISIS DE CALIDAD')
print('=' * 60)

# Nulos
nulos_por_col = df.isnull().sum()
pct_nulos = (nulos_por_col / len(df) * 100)
cols_con_nulos = (pct_nulos > 0).sum()
cols_criticos = (pct_nulos > 50).sum()

# Variables con más nulos
top_nulos = pct_nulos[pct_nulos > 0].sort_values(ascending=False).head(5)

# Tipos de datos
n_numericas = len(df.select_dtypes(include=[np.number]).columns)
n_categoricas = len(df.select_dtypes(include=['object', 'category']).columns)

print(f'📊 Variables numéricas: {n_numericas}')
print(f'🏷️ Variables categóricas: {n_categoricas}')
print(f'❓ Columnas con nulos: {cols_con_nulos}')
print(f'🔴 Columnas críticas (>50% nulos): {cols_criticos}')

ANÁLISIS DE CALIDAD
📊 Variables numéricas: 17
🏷️ Variables categóricas: 17
❓ Columnas con nulos: 14
🔴 Columnas críticas (>50% nulos): 2


In [4]:
# ============================================================================
# CELDA 4: GRÁFICO FLUJO DEL PROYECTO (dinámico)
# ============================================================================
# El gráfico se genera con la función centralizada generar_grafico_flujo()
# (src/html/estado_proyecto.py), que detecta el estado real de las 9 fases
# leyendo los ficheros señal en disco. Antes este gráfico tenía estados
# hardcodeados que quedaban obsoletos cada vez que se completaba una fase.
#
# El parámetro fase_actual='fase2' marca esta fase como "🔄 En curso" en
# el gráfico (color azul) — útil para los notebooks de conclusiones que
# están re-generando su propio HTML.
# ============================================================================

print('=' * 60)
print('GRÁFICO: FLUJO DEL PROYECTO (dinámico)')
print('=' * 60)

from src.html.estado_proyecto import generar_grafico_flujo

# Genera la figura Plotly leyendo el estado real de las 9 fases
fig_flujo = generar_grafico_flujo(fase_actual='fase2')

# Guardar el gráfico como HTML separado (lo embebe el iframe en celda 5)
fig_flujo.write_html(
    RUTA_GRAFICOS / 'm07_flujo_proyecto.html',
    include_plotlyjs='cdn'
)
print('✅ Gráfico de flujo generado dinámicamente (9 fases detectadas)')


GRÁFICO: FLUJO DEL PROYECTO (dinámico)


✅ Gráfico de flujo generado dinámicamente (9 fases detectadas)


In [5]:
# ============================================================================
# CELDA: GENERAR HTML
# ============================================================================

print('=' * 60)
print('GENERANDO HTML')
print('=' * 60)

# --- KPIs ---
n_vars_total = len(df.columns)
# Lectura dinámica de n_features desde metricas_modelo.json
# Antes este número estaba hardcoded a 19 (modelo D_strict obsoleto). Ahora
# se lee del JSON producido por f6_m00_preparacion (sistema dinámico).
import json as _json
_ruta_metricas = ROOT / 'data' / '06_evaluacion' / 'metricas_modelo.json'
if _ruta_metricas.exists():
    with open(_ruta_metricas, 'r', encoding='utf-8') as _f:
        _metricas = _json.load(_f)
    _n_features = _metricas.get('n_features', None)
    _n_features_t = _metricas.get('n_features_tecnicas', None)
    if _n_features is not None and _n_features_t is not None and _n_features != _n_features_t:
        n_vars_modelo_str = f'{_n_features} ({_n_features_t} técnicas)'
    elif _n_features is not None:
        n_vars_modelo_str = str(_n_features)
    else:
        n_vars_modelo_str = 'N/D'
else:
    n_vars_modelo_str = 'N/D'
tasa_abandono = df['abandono'].mean() * 100 if 'abandono' in df.columns else 29.2

KPIS = [
    {'valor': formato_numero_es(len(df)), 'titulo': 'Registros Analizados'},
    {'valor': str(n_vars_total), 'titulo': 'Variables Originales'},
    {'valor': n_vars_modelo_str, 'titulo': 'Features Modelo'},
    {'valor': f'{tasa_abandono:.1f}%', 'titulo': 'Tasa Abandono'},
]
kpis_html = generar_kpis_html(KPIS)

seccion_flujo = generar_seccion_html(
    titulo='Estado del Proyecto', icono='📍',
    contenido='<iframe src="graficos/m07_flujo_proyecto.html" width="100%" height="420" frameborder="0"></iframe>'
)


# --- Tabla hallazgos → decisiones Fase 3 ---
seccion_hallazgos = generar_seccion_html(
    titulo='Hallazgos EDA → Decisiones Fase 3', icono='🔗',
    contenido='''
    <p style="color:#4a5568; margin-bottom:16px;">
        Cada hallazgo relevante de esta fase se traduce en una decisión concreta
        en la Fase 3 (Feature Engineering). La tabla siguiente documenta esa trazabilidad.
    </p>
    <div style="overflow-x:auto;">
    <table style="width:100%; border-collapse:collapse; font-size:13px;">
        <thead>
            <tr style="background:#edf2f7;">
                <th style="padding:10px 12px; text-align:left; border-bottom:2px solid #e2e8f0;">Hallazgo (Fase 2)</th>
                <th style="padding:10px 12px; text-align:left; border-bottom:2px solid #e2e8f0;">Decisión (Fase 3)</th>
                <th style="padding:10px 12px; text-align:left; border-bottom:2px solid #e2e8f0;">Variable resultante</th>
            </tr>
        </thead>
        <tbody>
            <tr style="background:#fff;">
                <td style="padding:9px 12px; border-bottom:1px solid #e2e8f0;">cred_titulacion tiene varianza casi nula (240/300/384 créditos)</td>
                <td style="padding:9px 12px; border-bottom:1px solid #e2e8f0;">Excluida del modelo — sin poder predictivo</td>
                <td style="padding:9px 12px; border-bottom:1px solid #e2e8f0;">—</td>
            </tr>
            <tr style="background:#f7fafc;">
                <td style="padding:9px 12px; border-bottom:1px solid #e2e8f0;">nombre_trabajo y nombre_beca tienen NaN estructurales (~48% y ~47%)</td>
                <td style="padding:9px 12px; border-bottom:1px solid #e2e8f0;">Derivar variables binarias — no imputar</td>
                <td style="padding:9px 12px; border-bottom:1px solid #e2e8f0;">tiene_trabajo, tiene_beca</td>
            </tr>
            <tr style="background:#fff;">
                <td style="padding:9px 12px; border-bottom:1px solid #e2e8f0;">orden_preferencia tiene 12.6% NaN estructurales (sin preinscripción)</td>
                <td style="padding:9px 12px; border-bottom:1px solid #e2e8f0;">Analizar descriptivamente; excluida del modelo D_strict por leakage potencial</td>
                <td style="padding:9px 12px; border-bottom:1px solid #e2e8f0;">—</td>
            </tr>
            <tr style="background:#f7fafc;">
                <td style="padding:9px 12px; border-bottom:1px solid #e2e8f0;">cred_superados y cred_matriculados son acumulados (estructura longitudinal)</td>
                <td style="padding:9px 12px; border-bottom:1px solid #e2e8f0;">Agregar a nivel expediente: media, último valor, tasa de rendimiento</td>
                <td style="padding:9px 12px; border-bottom:1px solid #e2e8f0;">tasa_rendimiento, cred_superados_total</td>
            </tr>
            <tr style="background:#fff;">
                <td style="padding:9px 12px; border-bottom:1px solid #e2e8f0;">Alta cardinalidad en titulacion (40 valores) y pais_nombre (77 valores)</td>
                <td style="padding:9px 12px; border-bottom:1px solid #e2e8f0;">titulacion excluida del modelo; pais_nombre agrupado en vive_fuera (binaria)</td>
                <td style="padding:9px 12px; border-bottom:1px solid #e2e8f0;">vive_fuera, rama</td>
            </tr>
            <tr style="background:#f7fafc;">
                <td style="padding:9px 12px; border-bottom:1px solid #e2e8f0;">Crecimiento del 214.8% en matrículas (2010-2020); cohorte 2019-2020 atípica por COVID</td>
                <td style="padding:9px 12px; border-bottom:1px solid #e2e8f0;">Incluir curso_aca_ini (año de entrada) como feature temporal</td>
                <td style="padding:9px 12px; border-bottom:1px solid #e2e8f0;">curso_aca_ini</td>
            </tr>
            <tr style="background:#fff;">
                <td style="padding:9px 12px;">nota_acceso y nota_selectividad con ~9% y ~7% nulos (accesos sin nota)</td>
                <td style="padding:9px 12px;">Imputar con mediana por rama; crear flag de nulo</td>
                <td style="padding:9px 12px;">nota_acceso, flag_nota_acceso</td>
            </tr>
        </tbody>
    </table>
    </div>
    '''
)

contenido_html = kpis_html + seccion_flujo + seccion_hallazgos

# --- Generar HTML ---
html_completo = render_pagina_desde_fichero(
    'f2_m07_conclusiones.ipynb',
    contenido_html,
    carpeta_notebook='fase2_eda'
)
ruta_html = RUTA_FASE2 / "m07_conclusiones.html"
ruta_html.parent.mkdir(parents=True, exist_ok=True)
ruta_html.write_text(html_completo, encoding='utf-8')
print(f'✅ HTML generado: {ruta_html}')
print(f"\n✅ HTML: {ruta_html}")


GENERANDO HTML
✅ HTML generado: C:\FF\AU_UJI_v2\docs\html\fase2\m07_conclusiones.html

✅ HTML: C:\FF\AU_UJI_v2\docs\html\fase2\m07_conclusiones.html


In [6]:
# ============================================================================
# CELDA 6: RESUMEN FINAL
# ============================================================================

print('\n' + '=' * 60)
print('✅ F2-M07 COMPLETADO')
print('=' * 60)
print(f'\n📁 HTML: {ruta_html}')
print(f'📊 Gráfico: m07_flujo_proyecto.html')
print('\n' + '=' * 60)
print('🎉 FASE 2: EDA Datos Originales COMPLETADA')
print('=' * 60)
print('\nMódulos completados:')
print('   ✅ M00: Índice')
print('   ✅ M01: Inspección')
print('   ✅ M02: Calidad')
print('   ✅ M03: Nulos')
print('   ✅ M04: Univariante Numérico')
print('   ✅ M05: Univariante Categórico')
print('   ✅ M06: Evolución')
print('   ✅ M07: Conclusiones')
print('\n📌 Siguiente: Fase 3 - Feature Engineering')
print('=' * 60)


✅ F2-M07 COMPLETADO

📁 HTML: C:\FF\AU_UJI_v2\docs\html\fase2\m07_conclusiones.html
📊 Gráfico: m07_flujo_proyecto.html

🎉 FASE 2: EDA Datos Originales COMPLETADA

Módulos completados:
   ✅ M00: Índice
   ✅ M01: Inspección
   ✅ M02: Calidad
   ✅ M03: Nulos
   ✅ M04: Univariante Numérico
   ✅ M05: Univariante Categórico
   ✅ M06: Evolución
   ✅ M07: Conclusiones

📌 Siguiente: Fase 3 - Feature Engineering
